In [ ]:
from dateutil.relativedelta import relativedelta
import pandas as pd
from matplotlib.pyplot import title
from neuralprophet import NeuralProphet, set_log_level
import warnings
from statsmodels.graphics.tsaplots import plot_acf

warnings.simplefilter(action='ignore', category=FutureWarning)
set_log_level("ERROR")

data = pd.read_csv('./data/out_hourly.csv')
df = pd.DataFrame(data)
df['tsun'] = df['tsun'].fillna(0)
df_p = pd.DataFrame({
    'ds': df['timestamp'],
    'y': df['value'],
    'hum': df['rhum'],
    'sunrise': df['tsun'],
})
df_p['ds'] = pd.to_datetime(df_p['ds'])

df_events = df[['timestamp', 'event_name']]
df_events = df_events[df_events['event_name'].notna()]
df_events['ds'] = pd.to_datetime(df_events['timestamp']).dt.date
df_events = df_events.drop_duplicates(subset=['ds', 'event_name'])
df_events = df_events.drop('timestamp', axis=1)
df_chris = df_events[df_events['event_name'] == 'Christkindlemarkt']
df_events = df_events[df_events['event_name'] != 'Christkindlemarkt']
df_events['ds'] = pd.to_datetime(df_events['ds'])
date_list_events = df_events['ds'].tolist()
df_chris['ds'] = pd.to_datetime(df_chris['ds'])
date_list_chris = df_chris['ds'].tolist()

df_event = pd.DataFrame({
    "event": "dornbirn_events",
    "ds": date_list_events,
})
df_chri = pd.DataFrame({
    "event": "Christkindlemarkt",
    "ds": date_list_chris,
})



df_tag = df_p[(df_p['ds'] >= pd.to_datetime('2024-05-21')) & (df_p['ds'] < pd.to_datetime('2024-05-22'))]
df_tag.loc[:,'ds'] = df_tag['ds'].apply(lambda x: x - relativedelta(days=14))
df_p = pd.concat([df_p, df_tag], ignore_index=True)


df_tag2 = df_p[(df_p['ds'] >= pd.to_datetime('2024-08-10')) & (df_p['ds'] < pd.to_datetime('2024-08-11'))]
df_tag2.loc[:,'ds'] = df_tag2['ds'].apply(lambda x: x - relativedelta(days=7))
df_p = pd.concat([df_p, df_tag2], ignore_index=True)

df_tag3 = df_p[(df_p['ds'] >= pd.to_datetime('2024-07-28')) & (df_p['ds'] < pd.to_datetime('2024-07-29'))]
df_tag3.loc[:,'ds'] = df_tag3['ds'].apply(lambda x: x + relativedelta(days=7))
df_p = pd.concat([df_p, df_tag3], ignore_index=True)

df_p = df_p.sort_values('ds')
df_p = df_p.reset_index(drop=True)

df_p = df_p.set_index('ds')
full_range = pd.date_range(df_p.index.min(), df_p.index.max(), freq='H')
df_p = df_p.reindex(full_range)
df_p['y'] = df_p['y'].interpolate()  # lineare Interpolation
df_p = df_p.reset_index().rename(columns={'index': 'ds'})

"""
def tage_mit_fehlerhaften_stunden(df, ds_col='ds'):
    df = df.copy()
    df['date'] = df[ds_col].dt.date
    df['hour'] = df[ds_col].dt.hour
    stunden_pro_tag = df.groupby('date')['hour'].nunique()
    return stunden_pro_tag[stunden_pro_tag < 24]

print(tage_mit_fehlerhaften_stunden(df_p, ds_col='ds'))
"""


plt = df_p.plot(title= "Utilization", x="ds", y=['y', 'hum', 'sunrise'], figsize=(15, 5))


confidence_level = 0.9
boundaries = round((1 - confidence_level) / 2, 2)
# NeuralProphet only accepts quantiles value in between 0 and 1
quantiles = [boundaries, confidence_level + boundaries]

m = NeuralProphet(n_changepoints=10, yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, n_lags=168, quantiles=quantiles)

m.set_plotting_backend('plotly-static')
m = m.add_country_holidays("AT")
m.add_lagged_regressor('hum', n_lags=3)
m.add_lagged_regressor('sunrise', n_lags=2)
m.add_events('dornbirn_events')
m.add_events('Christkindlemarkt')
df_with_event = m.create_df_with_events(df_p, df_event)
df_with_events = m.create_df_with_events(df_with_event, df_chri)
# evaluate uncertainy on calibration set
train_df, cal_df = m.split_df(df_with_events, valid_p=0.1)
# Fit the model on the dataset
metrics = m.fit(df_with_events, freq='H', validation_df=cal_df, progress=None)

# Create a new dataframe reaching 365 into the future for our forecast, n_historic_predictions also shows historic data
df_future = m.make_future_dataframe(df_with_events, periods=30, n_historic_predictions=True)
forecast = m.predict(df_future)
#print(forecast)


m.highlight_nth_step_ahead_of_each_forecast(1).plot(forecast, figsize=(20, 8))
#m.plot(forecast, figsize=(20, 10))
print("Dataset size:", len(df_with_events))
print("Train dataset size:", len(train_df))
print("Validation dataset size:", len(cal_df))
method = "naive"  # or "cqr" for a more sophisticated method, see uncertainty tutorial
conformal_forecast = m.conformal_predict(train_df, cal_df, alpha=0.1, method=method)
m.highlight_nth_step_ahead_of_each_forecast(1).plot(conformal_forecast, figsize=(20, 10))

m.plot_parameters(components=['seasonality', 'autoregression', "lagged_regressors"])
m.plot_components(forecast, components=['seasonality', 'autoregression', "lagged_regressors"])

#The residuals are the difference between the model’s prediction and the real data. If the model is perfect, the residuals should be zero.
df_residuals = pd.DataFrame({"ds": df_with_events["ds"], "residuals": df_with_events["y"] - conformal_forecast["yhat1"]})
fig = df_residuals.plot(x="ds", y="residuals", figsize=(15, 8))

#autoregression
#plt = plot_acf(df_residuals["residuals"], lags=168)
